# 06 — Social charts (Pillow)

The publication-ready charts, each rendered **inline** (then saved to
`outputs/social/`). Workspace rule: `display()` first, then save — a chart is
never saved without being shown.

1. Share of box office earned home vs abroad (100% stacked bar)
2. All-time top domestic films, adjusted for inflation (dumbbell)
3. Top foreign-language films by U.S. box office (ranked bar, country sublabels)

Titles are **objective descriptions of what the chart shows** — no conclusions.
The subtitle carries additional relevant detail (method, scope, source year).

*(The genre 'share earned abroad' chart stays exploration-only in `04-viz` —
the genres are all too close in value to make a compelling social chart.)*

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import duckdb
from src.ingest import load_config
from chart_templates import lollipop, stacked_100pct_bars, single_ranked_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: viz notebooks only read, so they run alongside an open kernel.
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
img_w, img_h, _ = PRESETS['twitter_landscape']
web_w, web_h, _ = PRESETS['web']
out = Path(cfg['paths']['outputs_social']); out.mkdir(parents=True, exist_ok=True)
web_out = out.parent / 'web'; web_out.mkdir(parents=True, exist_ok=True)
def money(v):
    return f'${v/1e9:.2f}B' if abs(v) >= 1e9 else f'${v/1e6:.0f}M'

## 1. Share of box office earned home vs abroad
Top 15 U.S.-produced films by worldwide gross, each split into the share earned
at home (U.S. & Canada) vs the rest of the world, ordered by share abroad.

In [ ]:
# 100% stacked bar: each film's split home vs abroad, ordered by share abroad
# (the teal segment shrinks down the list, so the visual matches the sort).
ww = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE),
    top AS (SELECT w.title, w.release_year,
                   100.0*w.foreign_gross/w.worldwide_gross AS foreign_pct,
                   100.0*w.domestic_gross/w.worldwide_gross AS home_pct
            FROM films_worldwide w JOIN us ON us.title=w.title AND us.release_year=w.release_year
            ORDER BY w.worldwide_gross DESC LIMIT 15)
    SELECT * FROM top ORDER BY foreign_pct DESC''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
img1 = stacked_100pct_bars(ww, group_col='label',
    segments=[{'col':'foreign_pct','label':'Rest of world','color':'#005F73'},
              {'col':'home_pct','label':'Home (US/Canada)','color':'#EE9B00'}],
    title='Share of box office earned home vs abroad',
    subtitle='Top 15 U.S.-produced films by worldwide gross, ordered by share earned outside the U.S. & Canada.',
    source='Box Office Mojo, Top Lifetime Grosses (Worldwide) + TMDB origin country — as of Sep 2026',
    bar_height=34, bar_gap=12, img_width=img_w, img_height=img_h)
display(img1)
img1.save(out / '02_worldwide_domestic_vs_international.png')

## 2. All-time top domestic films, adjusted for inflation
Within the U.S. & Canada market, the biggest films once **ticket-price
inflation** is accounted for — Box Office Mojo's adjustment (estimated tickets
sold x today's average ticket price), i.e. an **admissions** basis. The gold dot
is what each film actually took at the time (nominal, release-year $); the teal
dot is the adjusted figure. This counts people through the door, the sound way to
compare box office across eras.

In [ ]:
dom = con.execute('''SELECT title, adjusted_gross, nominal_gross, release_year
    FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15''').df()
dom['label'] = dom['title'] + '  (' + dom['release_year'].astype(str) + ')'
img2 = lollipop(dom, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='All-time top 15 domestic films, adjusted for inflation',
    subtitle='U.S. & Canada gross, adjusted for ticket-price inflation (estimated tickets x today’s price). Gold dot = nominal, what each film made at the time.',
    source='Box Office Mojo, Top Lifetime Adjusted Grosses (domestic) — ticket-price adjusted, as of Sep 2026',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted (tickets x today’s price)', value2_label='Nominal (release $)', legend_reverse=True,
    img_width=img_w, img_height=img_h)
display(img2)
img2.save(out / '01_domestic_adjusted_vs_nominal.png')

## 3. Top foreign-language films by U.S. box office
The companion to chart 1: which non-English-language films earned the most in
the U.S. & Canada. Country of origin is shown under each title. Figures are
**nominal** (year-of-release dollars) — there is no reliable admissions
adjustment for this list, so older titles are modestly understated (noted on the
chart).

In [ ]:
foreign = con.execute('''SELECT rank_foreign, title, domestic_gross, release_year, origin_name
    FROM films_foreign_us ORDER BY domestic_gross DESC LIMIT 15''').df()
foreign['label'] = foreign['title'] + '  (' + foreign['release_year'].astype(str) + ')'
foreign['gross_label'] = foreign['domestic_gross'].apply(money)
img3 = single_ranked_bars(foreign, category_col='label', value_col='domestic_gross',
    total_label_col='gross_label', bar_color='#005F73', sublabel_col='origin_name',
    title='Top foreign-language films by U.S. box office',
    subtitle='Non-English-language films by U.S. & Canada lifetime gross (nominal $; older titles understated). Country of origin under each title.',
    source='Box Office Mojo (Foreign Language) + TMDB origin — nominal $, as of Sep 2026',
    img_width=img_w, img_height=img_h)
display(img3)
img3.save(out / '03_foreign_language_us_gross.png')

---
Three charts written to `outputs/social/`. Watermark `@unwelcomedata`, brand
palette, `twitter_landscape` preset.

## Web versions (for the project page)

Re-render each chart in **web mode** at the web preset: title/subtitle/source
dropped (the page's markdown carries them), only the `@unwelcomedata` watermark
kept, freed space handed to the data. These charts are built by direct template
calls (not `render_chart`), so each is re-called with `web_mode=True`. Web charts
\u2192 `outputs/web/`; social charts in `outputs/social/` are untouched.

In [ ]:
# Chart 1 (stacked) — web variant
img1w = stacked_100pct_bars(ww, group_col='label',
    segments=[{'col':'foreign_pct','label':'Rest of world','color':'#005F73'},
              {'col':'home_pct','label':'Home (US/Canada)','color':'#EE9B00'}],
    title='', subtitle='', source=None,
    bar_height=34, bar_gap=12, img_width=web_w, img_height=web_h, web_mode=True)
img1w.save(web_out / '02_worldwide_domestic_vs_international.png')

# Chart 2 (lollipop) — web variant
img2w = lollipop(dom, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='', subtitle='', source=None,
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted (tickets x today\u2019s price)', value2_label='Nominal (release $)', legend_reverse=True,
    img_width=web_w, img_height=web_h, web_mode=True)
img2w.save(web_out / '01_domestic_adjusted_vs_nominal.png')

# Chart 3 (ranked bars) — web variant
img3w = single_ranked_bars(foreign, category_col='label', value_col='domestic_gross',
    total_label_col='gross_label', bar_color='#005F73', sublabel_col='origin_name',
    title='', subtitle='', source=None,
    img_width=web_w, img_height=web_h, web_mode=True)
img3w.save(web_out / '03_foreign_language_us_gross.png')

print(f'\u2713 Web charts written to {web_out}')

## Cleanup
Close the DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')